## Step 2: Load and Inspect

In [132]:
import pandas as pd
df = pd.read_csv("../data/raw/netflix_titles.csv")

In [133]:
df.shape

(8807, 12)

In [134]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


Dataset contains 8,807 rows and 12 columns, matching the 12 expected fields (show_id, type, title, director, cast, country, date_added, release_year, rating, duration, listed_in, description).

In [135]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


**Missing values (from .info()):**
- director: 6,173 non-null out of 8,807 -> 2,634 missing (~30%)
- cast: 7,982 non-null -> 825 missing (~9%)
- country: 7,976 non-null -> 821 missing (~9%)
- date_added, racing, duration: negligible missingness (<15 rows each)

**Data types:** data_added and duration are both stored as 'str', not as real date or numeric type, despite looking like a date/number

In [136]:
(df.isnull().sum() / df.shape[0] * 100).sort_values(ascending=False)

director        29.908028
country          9.435676
cast             9.367549
date_added       0.113546
rating           0.045418
duration         0.034064
show_id          0.000000
type             0.000000
title            0.000000
release_year     0.000000
listed_in        0.000000
description      0.000000
dtype: float64

**Missing value % (sorted):**
- director: 29.9%
- country: 9.44%
- cast: 9.37%
- all other columns: <1%

In [137]:
df[df['type'] == 'Movie']['duration'].head(3)

0     90 min
6     91 min
7    125 min
Name: duration, dtype: str

In [138]:
df[df['type'] == 'TV Show']['duration'].head(3)

1    2 Seasons
2     1 Season
3     1 Season
Name: duration, dtype: str

**Structural findings:**
- 'type' has exactly two clean categories: Movie, TV Show.
- 'duration' means different things per type: "90 min" for Movies vs.
  "2 Seasons"/"1 Season" for TV Shows - will need to be split into a 
  numeric value + unit, conditional on type.
- 'listed_in' holds multiple comma-separated genres per row (e.g.
  "International TV Shows, TV Dramas, TV Mysteries") - will need to be 
  split into individual genres for genre-level analysis.

## Step 3: Cleaning

Based on the issues identified in Step 2, this section addresses four problems in order: missing values in director/cast/country, converting date_added to a real dat, splitting duration into a usable numeric value, and handling the multi-genre listed_in column.

In [139]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

In [140]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           4
duration         3
listed_in        0
description      0
dtype: int64

**Handling missing values in director, cast, country:**
Filled missing values with "Unknown" rather than dropping rows, since these columns are non-numeric and other analysis questions (e.g. titles per year, genre trends) don't depend on director/cast/country being present. Dropping ~30 of rows (the missingness in 'director' alone) would unnecessarily shrink the dataset for unrelated analyses.

In [141]:
df['date_added'] = df['date_added'].str.strip()
df['date_added'] = pd.to_datetime(df['date_added'])

In [142]:
df['date_added'].head()

0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-24
4   2021-09-24
Name: date_added, dtype: datetime64[us]

In [143]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      8807 non-null   str           
 4   cast          8807 non-null   str           
 5   country       8807 non-null   str           
 6   date_added    8797 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8803 non-null   str           
 9   duration      8804 non-null   str           
 10  listed_in     8807 non-null   str           
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), str(10)
memory usage: 825.8 KB


**Converting date_added to a real date:**
Some values had inconsistent leading whitespace (e.g. " August 4, 2027"), which broke pandas' single-format date parsing. Stripped whitespace first, then converted to datetime64 using pd.to_datetime(). The 10 originally-missing values remain as NaT (negligble, ~0.1% of rows) - left as-is rather than filled, since there's no sensible date to impute here.

In [144]:
df['duration'].str.split(' ')

0          [90, min]
1       [2, Seasons]
2        [1, Season]
3        [1, Season]
4       [2, Seasons]
            ...     
8802      [158, min]
8803    [2, Seasons]
8804       [88, min]
8805       [88, min]
8806      [111, min]
Name: duration, Length: 8807, dtype: object

In [145]:
df[df['duration'].isnull()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,2017-04-04,2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,2016-09-16,2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,2016-08-15,2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...


In [146]:
df.loc[5541]

show_id                                                     s5542
type                                                        Movie
title                                             Louis C.K. 2017
director                                               Louis C.K.
cast                                                   Louis C.K.
country                                             United States
date_added                                    2017-04-04 00:00:00
release_year                                                 2017
rating                                                     74 min
duration                                                      NaN
listed_in                                                  Movies
description     Louis C.K. muses on religion, eternal love, gi...
Name: 5541, dtype: object

In [147]:
shifted_rows = df['duration'].isnull()

In [148]:
df.loc[shifted_rows, 'duration'] = df.loc[shifted_rows, 'rating']

In [149]:
df.loc[shifted_rows, 'rating'] = 'Unknown'

In [150]:
df.loc[5541]

show_id                                                     s5542
type                                                        Movie
title                                             Louis C.K. 2017
director                                               Louis C.K.
cast                                                   Louis C.K.
country                                             United States
date_added                                    2017-04-04 00:00:00
release_year                                                 2017
rating                                                    Unknown
duration                                                   74 min
listed_in                                                  Movies
description     Louis C.K. muses on religion, eternal love, gi...
Name: 5541, dtype: object

In [151]:
df[df['rating'].isnull()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava ...,Unknown,"Oprah Winfrey, Ava DuVernay",Unknown,2017-01-26,2017,NaN,37 min,Movies,Oprah Winfrey sits down with director Ava DuVe...
6827,s6828,TV Show,Gargantia on the Verdurous Planet,Unknown,"Kaito Ishikawa, Hisako Kanemoto, Ai Kayano, Ka...",Japan,2016-12-01,2013,NaN,1 Season,"Anime Series, International TV Shows","After falling through a wormhole, a space-dwel..."
7312,s7313,TV Show,Little Lunch,Unknown,"Flynn Curry, Olivia Deeble, Madison Lu, Oisín ...",Australia,2018-02-01,2015,NaN,1 Season,"Kids' TV, TV Comedies","Adopting a child's perspective, this show take..."
7537,s7538,Movie,My Honor Was Loyalty,Alessandro Pepe,"Leone Frisa, Paolo Vaccarino, Francesco Miglio...",Italy,2017-03-01,2015,NaN,115 min,Dramas,"Amid the chaos and horror of World War II, a c..."


In [152]:
df['rating'].isnull().sum()

np.int64(4)

**Fixing shifted data (rating/duration/listed_in):**
Found 3 rows where source data had shifted one column: the real duration value ("74 min" etc.) was sitting in 'rating', and 'listed_in' was truncated to just "Movies". Recovered the duration value into the correct column, and set 'rating' to "Unknown" for these rows since the true rating was lost upstream. Confirmed no other missing ratings remain.

In [153]:
duration_split = df['duration'].str.split(' ')

In [154]:
df['duration_value'] = duration_split.str[0]

In [155]:
df['duration_value'].head()

0    90
1     2
2     1
3     1
4     2
Name: duration_value, dtype: object

In [156]:
df['duration_value'] = pd.to_numeric(df['duration_value'])

In [157]:
df[['duration', 'duration_value']].head(10)

,duration,duration_value
0,90 min,90
1,2 Seasons,2
2,1 Season,1
3,1 Season,1
4,2 Seasons,2
5,1 Season,1
6,91 min,91
7,125 min,125
8,9 Seasons,9
9,104 min,104


In [158]:
df['duration_value'].dtype

dtype('int64')

In [159]:
df['duration_unit'] = duration_split.str[1]

In [160]:
df['duration_unit'].unique()

array(['min', 'Seasons', 'Season'], dtype=object)

In [161]:
df[['type', 'duration', 'duration_value', 'duration_unit']].head(10)

,type,duration,duration_value,duration_unit
0,Movie,90 min,90,min
1,TV Show,2 Seasons,2,Seasons
2,TV Show,1 Season,1,Season
3,TV Show,1 Season,1,Season
4,TV Show,2 Seasons,2,Seasons
5,TV Show,1 Season,1,Season
6,Movie,91 min,91,min
7,Movie,125 min,125,min
8,TV Show,9 Seasons,9,Seasons
9,Movie,104 min,104,min


**Splitting 'duration' into 'duration_value' and 'duration_unit'::**
The original 'duration' column mixed two different units depending on 'type' ("90 min" for Movies, "2 Seasons"/"1 Season" for TV Shows), making it unusable for numeric analysis as-is. Split on the space character into a numeric 'duration_value' (int64, confirmed zero missing after fixing the shifted-column bug above) and a text 'duration_unit' (min/Season/Seasons).

Kept the original 'duration' column alongside the split columns, rather than dropping it, to preserve the raw soruce value for transparency - this lefts anyone reviewing the cleaning logic directly compare the parsed output against the original text without needing to re-derive it. 

In [162]:
df['genre_list'] = df['listed_in'].str.split(', ')

In [163]:
df['genre_list'].head()

0                                      [Documentaries]
1    [International TV Shows, TV Dramas, TV Mysteries]
2    [Crime TV Shows, International TV Shows, TV Ac...
3                             [Docuseries, Reality TV]
4    [International TV Shows, Romantic TV Shows, TV...
Name: genre_list, dtype: object

In [164]:
df['listed_in'].isnull().sum()

np.int64(0)

In [165]:
df_by_genre = df.explode('genre_list')

In [166]:
df['genre_list'].head()

0                                      [Documentaries]
1    [International TV Shows, TV Dramas, TV Mysteries]
2    [Crime TV Shows, International TV Shows, TV Ac...
3                             [Docuseries, Reality TV]
4    [International TV Shows, Romantic TV Shows, TV...
Name: genre_list, dtype: object

In [167]:
df['genre_list'].isnull().sum()

np.int64(0)

In [168]:
df_by_genre[['title', 'genre_list']].head(10)

,title,genre_list
0,Dick Johnson Is Dead,Documentaries
1,Blood & Water,International TV Shows
1,Blood & Water,TV Dramas
1,Blood & Water,TV Mysteries
2,Ganglands,Crime TV Shows
2,Ganglands,International TV Shows
2,Ganglands,TV Action & Adventure
3,Jailbirds New Orleans,Docuseries
3,Jailbirds New Orleans,Reality TV
4,Kota Factory,International TV Shows


In [169]:
df.shape

(8807, 15)

In [170]:
df_by_genre.shape

(19323, 15)

**Handling listed_in (multi-genre column):**
Split the comma-separated 'listed_in' string into a list column ('genre_list') using str.split(', '). Since genre-level analysis (top genres, genre trends) needs genre to behave as an individual category rather than a combined string, exploded genre_list into a separate DataFrame ('df_by_genre') with one row per title-genre pair, rather than overwriting the original 'df'. This preserves the one-row-per-title version for other analysis, while making df_by_genre available for genre-specific aggregation (e.g. count of titles per genre).

## Step 4: Explore with Pandas

Now that the data is cleaned, this section digs into a few concrete questions using pandas:

1. Has the number of titles added to Netflix grown year over year, and is that growth different for Movies vs. TV Shows?
2. What are the top 10 most common genres, and has that pattern been consistent over the years?
3. What's the average movie length, and is there a tendency toward longer, shorter, or middle-length movies?
4. Which countries contribute the most content to Netflix? (Note: "Unknown" is included as its own category here, since ~9% of country values were missing and dilled with "Unknown" in Step 3 - it's called out explicitly rather than silently dropped, so the missing-data context isn't lost.)

In [171]:
df['year_added'] = df['date_added'].dt.year

In [172]:
df['year_added'].value_counts()

year_added
2019.0    2016
2020.0    1879
2018.0    1649
2021.0    1498
2017.0    1188
2016.0     429
2015.0      82
2014.0      24
2011.0      13
2013.0      11
2012.0       3
2009.0       2
2008.0       2
2010.0       1
Name: count, dtype: int64

In [173]:
df['year_added'].dropna().value_counts().sort_index()

year_added
2008.0       2
2009.0       2
2010.0       1
2011.0      13
2012.0       3
2013.0      11
2014.0      24
2015.0      82
2016.0     429
2017.0    1188
2018.0    1649
2019.0    2016
2020.0    1879
2021.0    1498
Name: count, dtype: int64

In [174]:
df.groupby(['year_added', 'type']).size()

year_added  type   
2008.0      Movie         1
            TV Show       1
2009.0      Movie         2
2010.0      Movie         1
2011.0      Movie        13
2012.0      Movie         3
2013.0      Movie         6
            TV Show       5
2014.0      Movie        19
            TV Show       5
2015.0      Movie        56
            TV Show      26
2016.0      Movie       253
            TV Show     176
2017.0      Movie       839
            TV Show     349
2018.0      Movie      1237
            TV Show     412
2019.0      Movie      1424
            TV Show     592
2020.0      Movie      1284
            TV Show     595
2021.0      Movie       993
            TV Show     505
dtype: int64

In [175]:
year_type_counts = df.groupby(['year_added', 'type']).size().unstack()
year_type_counts

type,Movie,TV Show
year_added,,
2008.0,1.0,1.0
2009.0,2.0,NaN
2010.0,1.0,NaN
2011.0,13.0,NaN
2012.0,3.0,NaN
2013.0,6.0,5.0
2014.0,19.0,5.0
2015.0,56.0,26.0
2016.0,253.0,176.0


In [176]:
year_type_counts.index = year_type_counts.index.astype(int)
year_type_counts

type,Movie,TV Show
year_added,,
2008,1.0,1.0
2009,2.0,NaN
2010,1.0,NaN
2011,13.0,NaN
2012,3.0,NaN
2013,6.0,5.0
2014,19.0,5.0
2015,56.0,26.0
2016,253.0,176.0


**Q1: Growth in titles added over time (Movies vs. TV Shows)**

Titles added to Netflix grew substantially from 2008 through 2019, with the steepest growth between 2016-2019 (429 -> 1188 -> 1649 -> 2016 titles/year). Both 2020 and 2021 show a decline from the 2019 peak (1879, then 1498) - plausibly related to production slowdowns during COVID-19, though this dataset alone can't confirm causation.

Movies have outpaced TV Shows in raw count every single year, though TV Shows grew at a comparable rate - e.g. in 2019, 1424 Movies vs. 592 TV Shows were added, roughly a 2.4x ratio, fairly consistent across the later years.

Note: ~10 rows with missing 'date_added' are excluded from this year-by-year breakdown, since groupby drops rows with a missing group key by default.

In [177]:
df_by_genre.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'duration_value', 'duration_unit', 'genre_list'],
      dtype='str')

In [178]:
df_by_genre = df.explode('genre_list')
df_by_genre.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'duration_value', 'duration_unit', 'genre_list', 'year_added'],
      dtype='str')

In [179]:
top_genres = df_by_genre['genre_list'].value_counts()[:10]
top_genres

genre_list
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64

In [180]:
top_genre_names = top_genres.index
df_top_genres = df_by_genre[df_by_genre['genre_list'].isin(top_genre_names)]

In [181]:
df_top_genres.shape

(12708, 16)

In [182]:
genre_trend = df_top_genres.groupby(['year_added', 'genre_list']).size().unstack()
genre_trend.index = genre_trend.index.astype(int)
genre_trend = genre_trend.fillna(0).astype(int)
genre_trend

genre_list,Action & Adventure,Children & Family Movies,Comedies,Documentaries,Dramas,Independent Movies,International Movies,International TV Shows,Romantic Movies,TV Dramas
year_added,,,,,,,,,,
2008,0,0,0,0,1,1,0,0,0,0
2009,0,0,0,0,1,0,1,0,0,0
2011,0,1,0,0,13,0,1,0,0,0
2012,0,1,2,1,0,0,0,0,0,0
2013,0,2,1,1,0,0,0,1,0,4
2014,1,4,2,8,0,0,2,0,0,3
2015,2,11,12,13,12,7,10,3,1,8
2016,18,20,48,68,65,31,82,86,7,43
2017,97,78,177,206,293,116,395,205,63,130


**Q2: Top genres and consistency over time**

Top 10 genres overall: International Movies (2752), Dramas (2427), Comedies (1674), International TV Shows (1351), Documentaries (869), Action & Adventure (859), TV Dramas (763), Independent Movies (756), Children & Family Movies (641), Romantic Movies (616).

Though International Movies and Dramas show consistently high counts from 2016 onward, they don't peak in the same year: International Movies peaks in 2018 (668) and already begins declining in 2019 (610), a year ahead of Dramas, which peaks in 2019 (564) before declining. Most other top genres follow the Dramas pattern - growing through 2019, then declining in 2020-2021. Romantic Movies is a notable exception in the other direction, continuing to grow through 2020 (173) before dropping in 2021.

In [183]:
movies = df[df['type'] == 'Movie']

In [184]:
movies['duration_value'].mean()

np.float64(99.56499755341706)

In [185]:
movies['duration_unit'].unique()

array(['min'], dtype=object)

In [186]:
movies['duration_value'].describe()

count    6131.000000
mean       99.564998
std        28.289504
min         3.000000
25%        87.000000
50%        98.000000
75%       114.000000
max       312.000000
Name: duration_value, dtype: float64

In [187]:
movies[movies['duration_value'] == movies['duration_value'].min()][['title', 'duration_value', 'listed_in']]

,title,duration_value,listed_in
3777,Silent,3,"Children & Family Movies, Sci-Fi & Fantasy"


In [188]:
movies[movies['duration_value'] == movies['duration_value'].max()][['title', 'duration_value', 'listed_in']]

,title,duration_value,listed_in
4253,Black Mirror: Bandersnatch,312,"Dramas, International Movies, Sci-Fi & Fantasy"


**Q3: Average movie length and duration distribution**

Movies average around 99 minutes (median: 98 minutes), and there's a clear tendency toward middle-length films: the middle 50% of all movies fall between 87 and 114 minutes, a fairly tight cluster around standard feature-film runtime. The closeness of the mean and median suggests this figure is representative of the dataset as a whole, rather than skewed by outlets – even though a few extreme cases exist at the edges, like 'Black Mirror: Bandersnatch' (312 minutes, an interactive film whose length reflect its branching structure) and 'Silent' (3 minutes) - no clear explanation for this runtime is available from the dataset's other columns, though it may be a short film. 

In [189]:
df['country'].head(20)

0                                         United States
1                                          South Africa
2                                               Unknown
3                                               Unknown
4                                                 India
5                                               Unknown
6                                               Unknown
7     United States, Ghana, Burkina Faso, United Kin...
8                                        United Kingdom
9                                         United States
10                                              Unknown
11                                              Unknown
12                              Germany, Czech Republic
13                                              Unknown
14                                              Unknown
15                                        United States
16                                              Unknown
17                                              

In [190]:
df['country_list'] = df['country'].str.split(', ')
df_by_country = df.explode('country_list')

In [191]:
top_countries = df_by_country['country_list'].value_counts()[:10]
top_countries

country_list
United States     3689
India             1046
Unknown            831
United Kingdom     804
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Name: count, dtype: int64

In [192]:
top_countries_known = df_by_country[df_by_country['country_list'] != 'Unknown']['country_list'].value_counts()[:10]
top_countries_known

country_list
United States     3689
India             1046
United Kingdom     804
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Mexico             169
Name: count, dtype: int64

**Q4: Country contribution on Netflix:**

The United States leads by a wide margin (3689 country mentions), roughly 3.5x India's total (1046) - the two clear leaders, with a sharp drop-off after that. The rest of the top 10: United Kingdom (804), Canada (445), France (393), Japan (318), Spain (232), South Korea (231), Germany (226), and Mexico (169). 

Note: "Unknown" (831 entries, ~9% of country data) was excluded from this ranked list to avoid mixing missing data in with real countries - but it's worth noting it would rank 3rd overall, ahead of the UK, if include. 

Also note: these are country mentions, not unique titles - a co-produced title (e.g. one listing "United States, Ghana, Burkina Faso, United Kingdom") is counted once for each country listed, so these numbers reflect how many titles each country contributed to in some capacity, not titles produced exclusively by that country. 

## Step 5: Same Questions in SQL

Loaded the cleaned data into a local SQLite database (data/processed/netflix.db) using three tables: 'titles' (one row per title, for Q1/Q3), 'titles_by_genre' (one row per title-genre pair, for Q2), and 'titles_by_country' (one row per title-country pair, for Q4). List-type columns (genre_list, country_list) were excluded from tables that didn't need them, since SQLite can't store Python list objects directly.

In [193]:
import sqlite3
import os

In [194]:
os.makedirs('../data/processed', exist_ok=True)
conn = sqlite3.connect('../data/processed/netflix.db')

In [195]:
df_sql = df.drop(columns=['genre_list', 'country_list'])
df_by_genre = df.explode('genre_list')
df_by_country = df.explode('country_list')

In [196]:
df_sql.to_sql('titles', conn, if_exists='replace', index=False)
df_by_genre.drop(columns=['country_list']).to_sql('titles_by_genre', conn, if_exists='replace', index=False)
df_by_country.drop(columns=['genre_list']).to_sql('titles_by_country', conn, if_exists='replace', index=False)


DatabaseError: Execution failed

In [ ]:
print(pd.read_sql_query("SELECT COUNT(*) FROM titles", conn))
print(pd.read_sql_query("SELECT COUNT(*) FROM titles_by_genre", conn))
print(pd.read_sql_query("SELECT COUNT(*) FROM titles_by_country", conn))

In [ ]:
q1_sql = pd.read_sql_query("""
    SELECT
        strftime('%Y', date_added) AS year_added,
        type,
        COUNT(*) AS title_count
    FROM titles
    WHERE date_added IS NOT NULL
    GROUP BY year_added, type
    ORDER BY year_added;
""", conn)
q1_sql

**Q1 (SQL): Titles added per year, by type**

Rewrote the pandas year-by-type breakdown as a SQL query using strftime() to extract the year from date_added, GROUP BY year and type, with COUNT(*) for the title count. Results match the pandas analysis exactly (e.g. 2019: 1424 Movies / 592 TV Shows; 2008: 1 / 1), confirming the translation is correct.

In [ ]:
q2_sql = pd.read_sql_query("""
    SELECT
        genre_list AS genre,
        COUNT(*) AS title_count
    FROM titles_by_genre
    GROUP BY genre_list
    ORDER BY title_count DESC
    LIMIT 10;
""", conn)
q2_sql

**Q2 (SQL): Top 10 genres**

Rewrote the pandas genre ranking as a SQL query against titles_by_genre, using GROUP BY genre_list with COUNT(*), ORDER BY DESC, and LIMIT 10. Results match the pandas analysis exactly, confirming the translation (International Movies: 2752, Dramas: 2427, down through Romantic Movies: 616).

In [ ]:
q3_sql = pd.read_sql_query("""
    SELECT
        AVG(duration_value) AS avg_duration,
        MIN(duration_value) AS min_duration,
        MAX(duration_value) AS max_duration
    FROM titles
    WHERE type = 'Movie';
""", conn)
q3_sql

In [ ]:
q3_sql_percentiles = pd.read_sql_query("""
    WITH ranked AS (
        SELECT 
            duration_value,
            NTILE(4) OVER (ORDER BY duration_value) AS quartile
        FROM titles
        WHERE type = 'Movie'
    )
    SELECT 
        quartile,
        MIN(duration_value) AS min_val,
        MAX(duration_value) AS max_val,
        COUNT(*) AS n
    FROM ranked
    GROUP BY quartile;
""", conn)
q3_sql_percentiles

**Q3 (SQL): Average movie length and duration distribution**

Rewrote the pandas duration analysis in two parts. First, AVG/MIN/MAX against titles filtered to type = 'Movie' matched pandas exactly (avg: 99.56, min: 3, max: 312). Second, since SQLite has no built-in percentile function, used a window function (NTILE(4) OVER (ORDER BY duration_value)) inside a CTE to splot movies into quartiles, then took MIN/MAX per quartile as approximate percentile boundaries. The results matched pandas' 25th/50th/75th percentiles exactly: 87 / 98 / 114 minutes. 

In [ ]:
q4_sql = pd.read_sql_query("""
    SELECT
        country_list AS country,
        COUNT(*) AS title_count
    FROM titles_by_country
    WHERE country_list != 'Unknown'
    GROUP BY country_list
    ORDER BY title_count DESC
    LIMIT 10;
""", conn)
q4_sql

**Q4 (SQL): Top 10 countries**

Rewrote the pandas country ranking as a SQL query against titles_by_country (not titles, since the raw country column still holds comma-separated co-prodution strings that would undercount individual countries if queried directly). Used WHERE country_list != 'Unknown' to exclude the placeholder, GROUP BY/COUNT/ORDER BY DESC/LIMIT 10 for the ranking. Results match pandas exactly, from United States (3689) down to Mexico (169).

## Step 6: Export

Exported four CSVs to outputs/, ready for Tableau
- netflix_titles_cleaned.csv - full cleaned dataset (covers Q3 directly; Tableau can aggregate duration_value on its own for average/distribution)
- titles_by_year_and_type.csv - Q1 summary (titles per year, Movie vs TV Show)
- genre_trend_by_year.csv - Q2 Summary (top genres per year)
- top_countries.csv - Q4 summary (top 10 countries by content count)

In [197]:
df.to_csv('../outputs/netflix_titles_cleaned.csv', index=False)
year_type_counts.to_csv('../outputs/titles_by_year_and_type.csv', index=False)
genre_trend.reset_index().to_csv('../outputs/genre_trend_by_year.csv', index=False)
top_countries_known.reset_index().to_csv('../outputs/top_countries.csv', index=False)